In [1]:
import re
import pandas as pd
import numpy as np
import nltk
import pymorphy3
import optuna
import mlflow
import pickle
import implicit

from implicit.als import AlternatingLeastSquares
from scipy.sparse import coo_matrix, csr_matrix
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from nltk.corpus import stopwords
from mlxtend.plotting import plot_decision_regions
from tqdm import tqdm
from scipy.sparse import csr_matrix, vstack
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from gensim.utils import simple_preprocess
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.simplefilter('ignore', FutureWarning)

from sklearn import set_config
set_config(display='diagram')

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "hr-ai-scout"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

# Load data

In [4]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_salary_from,vacancy_salary_to,vacancy_salary_currency,vacancy_salary_gross,...,resume_education,resume_courses,resume_salary,resume_age,resume_total_experience,resume_experience_months,resume_location,resume_gender,resume_applicant_status,target
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Казанский Авиационный Институт'],NaN,NaN,65.0,19 лет,228.0,Москва,Мужчина,Рассматривает предложения,1
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,"['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...","['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...",NaN,43.0,17 лет 4 месяца,208.0,Москва,Мужчина,Рассматривает предложения,1
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Орский государственный педагогический инстит...,NaN,200 000 ₽ на руки,52.0,30 лет,360.0,Москва,Женщина,NaN,1
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Красноярский государственный университет'],NaN,500 000 ₽ на руки,56.0,29 лет 8 месяцев,356.0,Красноярск,Мужчина,Рассматривает предложения,1
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Белоруский Гос. Университет Информатики и Ра...,"['SAP CIS, SAP XI', 'Школа Логистики МАДИ', 'S...",NaN,48.0,25 лет 1 месяц,301.0,Moscow,Male,NaN,1


# Preprocessing

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
В первую очередь уберем строки, где пропущены все ключевые значения в резюме:
</div>

In [5]:
t1 = df.shape[0]
df = df.dropna(subset= ["resume_education",
                        "resume_last_experience_description",
                        "resume_last_position",
                        "resume_last_company_experience_period",
                        "resume_total_experience",
                        "resume_experience_months",
                        "resume_location",
                        "resume_specialization",
                        # "resume_gender",
                        # "resume_title"
                       ], how="all")
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строки')

Удалено  84  строки


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Удалим еще те строки, где случился технический сбой в парсинге, где у кандидата общий опыт есть, а последний опыт не указан (и наоборот):
</div>

In [6]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
        & df["resume_last_experience_description"].isna()
        & df["resume_last_position"].isna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  1543  строк


In [7]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].isna()
        & df["resume_last_experience_description"].notna()
        & df["resume_last_position"].notna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  0  строк


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Посмотрим на пропуски отдельно по категориальным и числовым признакам.
</div>

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [9]:
df[cat_cols] = df[cat_cols].fillna('NDT')

In [10]:
df.loc[df['resume_experience_months'].isna(), 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

In [11]:
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)

In [12]:
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)

In [13]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Преобразуем сначала ожидаемые зарплаты
</div>

In [14]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())

df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(part for part in x if re.fullmatch(r'\d+', part)))
              if any(re.fullmatch(r'\d+', part) for part in x)
              else np.nan
)

currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]

rates_rub = {
    "₽": 1.0,
    "$": 80.85,
    "€": 94.14,
    "₴": 1.94,
    "₸": 0.150,
    "₼": 47.8,
    "₾": 33.5,
    "Br": 28.7,
    "so'm": 0.0068
}

df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((sym for sym in x if sym in currency_symbols), np.nan)
)

df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)

df['resume_salary'] = df['salary_converted']

df = df.drop(['resume_salary_split', 'salary_int', 'currency_symbol', 'salary_converted'], axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим дополнительный столбец с опытом работы в последней компании в месяцах для удобства
</div>

In [15]:
def experience_to_months(experience_text):
    months = 0
    # Опыт в годах
    years_match = re.search(r'(\d+)\s*год', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    years_match = re.search(r'(\d+)\s*лет', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    # Опыт в месяцах
    months_match = re.search(r'(\d+)\s*месяц', experience_text)
    if months_match:
        months += int(months_match.group(1))

    return months if months > 0 else np.nan

In [16]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_period'].apply(experience_to_months)

In [17]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Т.к. в названии компании стоит NDT, можно столбец resume_last_company_experience_months заполнять нулевыми значениями.
</div>

In [18]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_months'].fillna(0)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Ограничим выбросы по зарплате, потому что ровно одно значение по ожидаемой заработоной плате = 999,999,999 (смешно, но нет)

- Ограничим опыт общий и внутри одной компании до 720 месяцев (60 лет, ничего себе уже)

- Уберем возраст > 90, не ждем, что эти кандидаты находятся в поиске вакансии
</div>

In [19]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Также уберем строки, где последний опыт кандидата больше, чем общий

- И где общий опыт кандидата +16 лет больше чем возраст (хоть так)

</div>

In [20]:
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Заменим текущий формат разброса полов в датасете на унифицированный

</div>

In [21]:
gender_map = {
    'Мужчина': 'Мужчина',
    'Male': 'Мужчина',
    'Женщина': 'Женщина',
    'Female': 'Женщина'
}

df['resume_gender'] = df['resume_gender'].apply(lambda x: gender_map[x] if x in gender_map else 'Неизвестно')

# Feature engineering

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим признак матчинга локации вакансии и резюме

</div>

In [22]:
df['location_matching'] = df.apply(lambda row: 1 if row['vacancy_area'] == row['resume_location'] else 0, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сделаем новый признак, а именно посчитаем количество навыков кандидата, которые указаны в вакансии.

</div>

In [23]:
def resume_skill_count_in_vacancy(row):
    count = 0
    skill_list = row['resume_skills'].replace('[', '').replace(']', '').replace("'", "").split(', ')
    for i in skill_list:
        if i in row['vacancy_description']:
            count += 1
    return count

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Также посчитаем долю слов из последней должности в резюме, которые указаны в вакансии.

</div>

In [24]:
def last_position_in_vacancy(row):
    bow = []
    seps = [' ', '-', '_']
    for sep in seps:
        bow += row['resume_last_position'].split(sep=sep)
        bow = list(set(bow))
    
    c = 0
    for word in bow:
        if word in row['vacancy_description']:
            c +=1
    
    return c / len(bow)

In [25]:
df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь закодируем описание вакансии и последнего опыта работы и сравним через косинусное расстояние.

</div>

In [26]:
def preprocess_data(df):
    """Обработка пропущенных значений в текстовых полях"""
    print("Проверка пропущенных значений...")
    print(f"Пропуски в vacancy_description: {df['vacancy_description'].isna().sum()}")
    print(f"Пропуски в resume_last_experience_description: {df['resume_last_experience_description'].isna().sum()}")
    
    # Заполняем пропуски пустыми строками
    df['vacancy_description'] = df['vacancy_description'].fillna('')
    df['resume_last_experience_description'] = df['resume_last_experience_description'].fillna('')
    
    # Проверяем, что все значения теперь строковые
    df['vacancy_description'] = df['vacancy_description'].astype(str)
    df['resume_last_experience_description'] = df['resume_last_experience_description'].astype(str)
    
    return df

In [27]:
def save_results(df, output_file):
    """Сохранение результатов в CSV файл"""
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Результаты сохранены в файл: {output_file}")

In [28]:
def calculate_cosine_similarity(embeddings1, embeddings2):
    """Вычисление косинусного сходства между двумя наборами эмбеддингов"""
    similarities = []
    
    for i in tqdm(range(embeddings1.shape[0])):
        emb1_row = embeddings1[i]
        emb2_row = embeddings2[i]
        
        similarity = cosine_similarity(emb1_row, emb2_row)[0][0]
        similarities.append(similarity)
    
    return similarities

In [29]:
warnings.filterwarnings('ignore')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('taggers/averaged_perceptron_tagger_ru')
except LookupError:
    nltk.download('averaged_perceptron_tagger_ru')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

morph = pymorphy3.MorphAnalyzer()

[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [30]:
def lemmatize_russian(tokens):
    """Лемматизация русских слов"""
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]  # Берем самый вероятный разбор
        lemmas.append(parsed.normal_form)
    return lemmas

In [31]:
def tokenize_and_lemmatize(text):
    """Токенизация текста с лемматизацией и удалением стоп-слов"""
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    stop_words = set(stopwords.words('russian') + stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    lemmatized_tokens = lemmatize_russian(tokens)
    
    return lemmatized_tokens

In [32]:
def get_tfidf_embeddings(texts, vectorizer=None, fit=True):
    """Создание TF-IDF эмбеддингов для списка текстов с лемматизацией"""
    if fit:
        vectorizer = TfidfVectorizer(
            max_features=5000,
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2),
            tokenizer=tokenize_and_lemmatize,
            token_pattern=None,
            lowercase=False  # Уже сделано в токенизации
        )
        embeddings = vectorizer.fit_transform(texts)
    else:
        embeddings = vectorizer.transform(texts)
    
    return embeddings, vectorizer

In [33]:
def get_tfidf_vacancy_embeddings(df, vectorizer=None):
    """Создание эмбеддингов для уникальных вакансий с лемматизацией"""
    unique_vacancies = df[['vacancy_id', 'vacancy_description']].drop_duplicates()
    
    unique_embeddings, vectorizer = get_tfidf_embeddings(
        unique_vacancies['vacancy_description'].tolist(), 
        vectorizer=vectorizer, 
        fit=(vectorizer is None)
    )
    
    vacancy_embedding_dict = dict(zip(unique_vacancies['vacancy_id'], unique_embeddings))
    
    rows = []
    for vid in df['vacancy_id']:
        rows.append(vacancy_embedding_dict[vid])
    
    all_vacancy_embeddings = vstack(rows)
    
    return all_vacancy_embeddings, vectorizer

In [34]:
def process_similarity_scores_tfidf(df, vectorizer=None, fit=True):
    """Функция для вычисления схожести с использованием TF-IDF и лемматизации"""
    df = preprocess_data(df)
    
    print("Создание TF-IDF эмбеддингов для описаний опыта в резюме...")
    experience_embeddings, tfidf_vectorizer = get_tfidf_embeddings(df['resume_last_experience_description'].tolist(), vectorizer=vectorizer, fit=fit)
    
    print("Создание TF-IDF эмбеддингов для описаний вакансий...")
    vacancy_embeddings, _ = get_tfidf_vacancy_embeddings(df, vectorizer=tfidf_vectorizer)
    
    print("Вычисление косинусного сходства...")
    similarity_scores = calculate_cosine_similarity(vacancy_embeddings, experience_embeddings)
    
    df['similarity_score_tfidf'] = similarity_scores
    
    return df, tfidf_vectorizer

In [35]:
try:
    df_tfidf = pd.read_csv('description_df_with_scores_tfidf.csv')
except:
    df_tfidf = process_similarity_scores_tfidf(df.copy())
    save_results(df_tfidf, 'description_df_with_scores_tfidf.csv')

In [36]:
df = df.merge(df_tfidf)

In [37]:
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_description,resume_id,resume_title,resume_specialization,...,resume_location,resume_gender,resume_applicant_status,target,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,resume_skill_count,similarity_score_tfidf
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",6969174,ABAP-разработчик,"['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,76.0,1,3,0.666667,3,0.284047
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",9100077,"ABAP разработчик - SAP HCM, CRM, S/4HANA ERP(F...","['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,8.0,1,2,0.500000,2,0.308726
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",32644957,Разработчик ABAP,"['Программист, разработчик']",...,Москва,Женщина,NDT,1,136.0,1,1,0.000000,1,0.510093
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",27220466,ABAP-разработчик,"['Программист, разработчик']",...,Красноярск,Мужчина,Рассматривает предложения,1,135.0,0,2,0.333333,2,0.301062
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",7532708,ABAP разработчик. Senior ABAP Developer. SAP T...,"['Programmer, developer']",...,Moscow,Мужчина,NDT,1,0.0,0,2,0.600000,2,0.075429


# Train-test split

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Добавим новые признаки в обучение и тестирование

</div>

In [38]:
features = [
    'vacancy_area',
    'vacancy_experience',
    'vacancy_employment', 
    'vacancy_schedule',
    # 'resume_specialization',
    # 'resume_education', 
    # 'resume_courses', 
    'resume_salary',
    'resume_age', 
    'resume_experience_months',
    'resume_location',
    'resume_gender', 
    'resume_applicant_status', 
    'resume_last_company_experience_months', 
    'location_matching',
    'resume_skill_count_in_vacancy',
    'last_position_in_vacancy',
    'similarity_score_tfidf'
]
df[features]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf
0,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,65.000000,228.0,Москва,Мужчина,Рассматривает предложения,76.0,1,3,0.666667,0.284047
1,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,43.000000,208.0,Москва,Мужчина,Рассматривает предложения,8.0,1,2,0.500000,0.308726
2,Москва,Более 6 лет,Полная занятость,Удаленная работа,200000.0,52.000000,360.0,Москва,Женщина,NDT,136.0,1,1,0.000000,0.510093
3,Москва,Более 6 лет,Полная занятость,Удаленная работа,500000.0,56.000000,356.0,Красноярск,Мужчина,Рассматривает предложения,135.0,0,2,0.333333,0.301062
4,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,48.000000,301.0,Moscow,Мужчина,NDT,0.0,0,2,0.600000,0.075429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321637,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,242550.0,66.000000,521.0,Санкт-Петербург,Женщина,NDT,270.0,0,0,0.166667,0.072670
321638,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,40.000000,213.0,Москва,Мужчина,Активно ищет работу,35.0,1,0,0.000000,0.000000
321639,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,80000.0,44.060813,121.0,Москва,Мужчина,NDT,44.0,1,0,0.200000,0.047398
321640,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,32.000000,117.0,Москва,Женщина,NDT,96.0,1,0,0.200000,0.029086


In [39]:
numeric_features = df[features].select_dtypes(include=np.number).columns
categorical_features = df[features].select_dtypes(exclude=np.number).columns

In [40]:
X = df[features].copy()
y = df['target'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [41]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((257313, 15), (64329, 15), (257313,), (64329,))

In [42]:
def calculate_metrics(df_test: pd.DataFrame) -> pd.DataFrame:
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    vacancy_ids = df_test['vacancy_id'].unique()
    
    for vacancy_id in vacancy_ids:
        mask = df_test['vacancy_id'] == vacancy_id
        y_true = df_test.loc[mask, 'target'].values
        y_score = df_test.loc[mask, 'y_pred_proba'].values
        
        if len(y_true) <= 1:
            continue
        
        y_true_2d = y_true.reshape(1, -1)
        y_score_2d = y_score.reshape(1, -1)
        
        ndcg = ndcg_score(y_true_2d, y_score_2d)
        ndcg_scores.append(ndcg)
        
        y_pred_binary = (y_score >= 0.5).astype(int)
        
        precision = precision_score(y_true, y_pred_binary, zero_division=0)
        recall = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
    
    if ndcg_scores:
        print(f"Средний NDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Средний Precision: {np.mean(precision_scores):.4f}")
        print(f"Средний Recall: {np.mean(recall_scores):.4f}")
        print(f"Средний F1-Score: {np.mean(f1_scores):.4f}")

        return np.mean(ndcg_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(f1_scores)
    else:
        print("Недостаточно данных для расчета метрик")

        return None, None, None, None

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Обучим с подбором гиперпараметров `LogisticRegression`, как бейзлайн для сравнения с нелинейными моделями, `RandomForestClassifier`, как разновидность бэггинга, и три разные вида градиентного бустинга: `XGBClassifier`, `LGBMClassifier` и `CatBoostClassifier`.

</div>

# LogisticRegression

In [43]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__C': trial.suggest_float('C', 1e-4, 1e4, log=True),
        'model__penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'model__solver': trial.suggest_categorical('solver', ['liblinear', 'saga'])
    }
    
    pipeline_lr_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
        ])),
        ('model', LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_lr_optuna.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lr_optuna.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lr_optuna.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [44]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'LogisticRegression_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run LogisticRegression_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/f4610617e52f411594ea97419dc2808c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [45]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "LogisticRegression_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [46]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=True)
study.optimize(objective, 
               n_trials=10, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-03-09 14:28:14,048] Using an existing study with name 'LogisticRegression_optuna' instead of creating a new one.


Средний NDCG: 0.8296
Средний Precision: 0.5714
Средний Recall: 0.7891
Средний F1-Score: 0.6323
Средний NDCG: 0.8179
Средний Precision: 0.5629
Средний Recall: 0.7775
Средний F1-Score: 0.6213


[I 2026-03-09 14:28:49,553] Trial 31 finished with value: 0.8259966050728942 and parameters: {'C': 0.030957271678029096, 'penalty': 'l1', 'solver': 'liblinear'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8260
Средний Precision: 0.5708
Средний Recall: 0.7824
Средний F1-Score: 0.6302
🏃 View run 31 at: http://127.0.0.1:5000/#/experiments/1/runs/3fd356772fef4bd1be53d59815464e49
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5698
Средний Recall: 0.7900
Средний F1-Score: 0.6316
Средний NDCG: 0.8190
Средний Precision: 0.5620
Средний Recall: 0.7803
Средний F1-Score: 0.6224


[I 2026-03-09 14:30:40,170] Trial 32 finished with value: 0.8267499383547057 and parameters: {'C': 0.6814921768639968, 'penalty': 'l1', 'solver': 'liblinear'}. Best is trial 30 with value: 0.8272130448068064.


...

Средний NDCG: 0.8269
Средний Precision: 0.5707
Средний Recall: 0.7814
Средний F1-Score: 0.6298
🏃 View run 38 at: http://127.0.0.1:5000/#/experiments/1/runs/90f304dcf2e64655961a726c6ed15356
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8299
Средний Precision: 0.5677
Средний Recall: 0.7879
Средний F1-Score: 0.6292
Средний NDCG: 0.8188
Средний Precision: 0.5617
Средний Recall: 0.7793
Средний F1-Score: 0.6219


[I 2026-03-09 14:45:41,475] Trial 39 finished with value: 0.826495616279113 and parameters: {'C': 10.49889426767395, 'penalty': 'l1', 'solver': 'liblinear'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8265
Средний Precision: 0.5705
Средний Recall: 0.7817
Средний F1-Score: 0.6297
🏃 View run 39 at: http://127.0.0.1:5000/#/experiments/1/runs/30fb0c6133b4485a83d59678834f2298
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8303
Средний Precision: 0.5691
Средний Recall: 0.7891
Средний F1-Score: 0.6307
Средний NDCG: 0.8191
Средний Precision: 0.5629
Средний Recall: 0.7800
Средний F1-Score: 0.6225


[I 2026-03-09 14:46:50,822] Trial 40 finished with value: 0.8270282741092976 and parameters: {'C': 0.2783728179309043, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8270
Средний Precision: 0.5715
Средний Recall: 0.7821
Средний F1-Score: 0.6306
🏃 View run 40 at: http://127.0.0.1:5000/#/experiments/1/runs/14acbf52b2f84053b4ae20468f5a0114
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 41
Best params: {'C': 0.18851801168357057, 'penalty': 'l1', 'solver': 'liblinear'}


In [47]:
study.best_trial.user_attrs['pipeline_params']

{'model__C': 0.18851801168357057,
 'model__penalty': 'l1',
 'model__solver': 'liblinear'}

In [48]:
pipeline_lr_best_optuna = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('numeric_scaling', StandardScaler(), numeric_features),
        ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

pipeline_lr_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_lr_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [49]:
y_pred_proba = pipeline_lr_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [50]:
ndcg_lr, precision_lr, recall_lr, f1_lr = calculate_metrics(df_test)

metrics_lr = {
    'NDCG': ndcg_lr,
    'Precision': precision_lr,
    'Recall': recall_lr,
    'F1': f1_lr
}

Средний NDCG: 0.7515
Средний Precision: 0.6386
Средний Recall: 0.5985
Средний F1-Score: 0.6021


In [51]:
RUN_NAME = "best_optuna_lr" 
REGISTRY_MODEL_NAME = "best_optuna_lr"

In [52]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_lr_best_optuna, 
                                       artifact_path='best_optuna_lr',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_lr)
    mlflow.log_params(best_params_optuna)

Registered model 'best_optuna_lr' already exists. Creating a new version of this model...
2026/03/09 14:47:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lr, version 4


🏃 View run best_optuna_lr at: http://127.0.0.1:5000/#/experiments/1/runs/535ce4b7dfab44fb9669ca60134aa9c0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_lr'.


# RandomForestClassifier

In [58]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 30),
        'model__min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'model__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20)
    }
    
    pipeline_rf = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_rf.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_rf.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_rf.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [59]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'RandomForestClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run RandomForestClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/9d840a31ab0443ebb1ff9c08816132a2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [60]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "RandomForestClassifier_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [61]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=True)
study.optimize(objective, 
               n_trials=10, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-03-09 14:55:44,879] Using an existing study with name 'RandomForestClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8554
Средний Precision: 0.6878
Средний Recall: 0.8307
Средний F1-Score: 0.7301
Средний NDCG: 0.8443
Средний Precision: 0.6779
Средний Recall: 0.8195
Средний F1-Score: 0.7200


[I 2026-03-09 14:58:17,606] Trial 27 finished with value: 0.8515895720812053 and parameters: {'n_estimators': 450, 'max_depth': 26, 'min_samples_split': 17, 'min_samples_leaf': 14}. Best is trial 24 with value: 0.8533184913910616.


Средний NDCG: 0.8516
Средний Precision: 0.6856
Средний Recall: 0.8253
Средний F1-Score: 0.7271
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/1/runs/b4ba0404eb084c32b57fa67e7c946f52
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8566
Средний Precision: 0.6904
Средний Recall: 0.8299
Средний F1-Score: 0.7317
Средний NDCG: 0.8444
Средний Precision: 0.6819
Средний Recall: 0.8184
Средний F1-Score: 0.7221


[I 2026-03-09 15:00:53,564] Trial 28 finished with value: 0.8517728720243215 and parameters: {'n_estimators': 450, 'max_depth': 25, 'min_samples_split': 17, 'min_samples_leaf': 13}. Best is trial 24 with value: 0.8533184913910616.


...

Средний NDCG: 0.8535
Средний Precision: 0.7040
Средний Recall: 0.8204
Средний F1-Score: 0.7373
🏃 View run 34 at: http://127.0.0.1:5000/#/experiments/1/runs/47a653cb98c64d3baed7b9e85c477617
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8510
Средний Precision: 0.6744
Средний Recall: 0.8258
Средний F1-Score: 0.7189
Средний NDCG: 0.8389
Средний Precision: 0.6577
Средний Recall: 0.8151
Средний F1-Score: 0.7041


[I 2026-03-09 15:25:01,075] Trial 35 finished with value: 0.8474410345915465 and parameters: {'n_estimators': 800, 'max_depth': 11, 'min_samples_split': 20, 'min_samples_leaf': 7}. Best is trial 34 with value: 0.8534553059658585.


Средний NDCG: 0.8474
Средний Precision: 0.6688
Средний Recall: 0.8226
Средний F1-Score: 0.7149
🏃 View run 35 at: http://127.0.0.1:5000/#/experiments/1/runs/d0cd13447186465cb45f5cb6fd2ba7bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8566
Средний Precision: 0.6985
Средний Recall: 0.8284
Средний F1-Score: 0.7366
Средний NDCG: 0.8453
Средний Precision: 0.6879
Средний Recall: 0.8181
Средний F1-Score: 0.7260


[I 2026-03-09 15:28:48,394] Trial 36 finished with value: 0.8528192314334716 and parameters: {'n_estimators': 700, 'max_depth': 28, 'min_samples_split': 19, 'min_samples_leaf': 11}. Best is trial 34 with value: 0.8534553059658585.


Средний NDCG: 0.8528
Средний Precision: 0.6942
Средний Recall: 0.8230
Средний F1-Score: 0.7319
🏃 View run 36 at: http://127.0.0.1:5000/#/experiments/1/runs/a08c993eb1b6493fb839ee637682a0ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 37
Best params: {'n_estimators': 750, 'max_depth': 27, 'min_samples_split': 20, 'min_samples_leaf': 8}


In [62]:
study.best_trial.user_attrs['pipeline_params']

{'model__n_estimators': 750,
 'model__max_depth': 27,
 'model__min_samples_split': 20,
 'model__min_samples_leaf': 8}

In [63]:
pipeline_rf_best_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])

pipeline_rf_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_rf_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [64]:
y_pred_proba = pipeline_rf_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [65]:
ndcg_rf, precision_rf, recall_rf, f1_rf = calculate_metrics(df_test)

metrics_rf = {
    'NDCG': ndcg_rf,
    'Precision': precision_rf,
    'Recall': recall_rf,
    'F1': f1_rf
}

Средний NDCG: 0.7755
Средний Precision: 0.6568
Средний Recall: 0.7464
Средний F1-Score: 0.6813


In [66]:
RUN_NAME = "best_optuna_rf" 
REGISTRY_MODEL_NAME = "best_optuna_rf"

In [67]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_rf_best_optuna, 
                                       artifact_path='best_optuna_rf',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_rf)
    mlflow.log_params(best_params_optuna)

Registered model 'best_optuna_rf' already exists. Creating a new version of this model...
2026/03/09 15:31:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_rf, version 3


🏃 View run best_optuna_rf at: http://127.0.0.1:5000/#/experiments/1/runs/8c413187913044e2a5200f078a65dba6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '3' of model 'best_optuna_rf'.


# XGBClassifier

In [68]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [69]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/9ce6ac17418b488bbba4bbe96338b15c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [70]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME_XGB = "XGBClassifier_optuna"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [71]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=True)

study_xgb.optimize(objective_xgb, 
                   n_trials=10, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-03-09 15:31:23,011] Using an existing study with name 'XGBClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8612
Средний Precision: 0.7589
Средний Recall: 0.8167
Средний F1-Score: 0.7710
Средний NDCG: 0.8478
Средний Precision: 0.7464
Средний Recall: 0.8034
Средний F1-Score: 0.7582


[I 2026-03-09 15:31:52,925] Trial 20 finished with value: 0.8577213553167893 and parameters: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.09389339222474813, 'min_child_weight': 10}. Best is trial 20 with value: 0.8577213553167893.


Средний NDCG: 0.8577
Средний Precision: 0.7538
Средний Recall: 0.8087
Средний F1-Score: 0.7650
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/1/runs/45734cabf7ba4ec7a43ca73cd0d34e21
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8610
Средний Precision: 0.7596
Средний Recall: 0.8159
Средний F1-Score: 0.7707
Средний NDCG: 0.8478
Средний Precision: 0.7489
Средний Recall: 0.8043
Средний F1-Score: 0.7598


[I 2026-03-09 15:32:23,192] Trial 21 finished with value: 0.8583336583209565 and parameters: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.09866580252678012, 'min_child_weight': 10}. Best is trial 21 with value: 0.8583336583209565.


...

Средний NDCG: 0.8568
Средний Precision: 0.7358
Средний Recall: 0.8213
Средний F1-Score: 0.7589
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/1/runs/f198854fa69d4e53b1220ec3a28627c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8597
Средний Precision: 0.7674
Средний Recall: 0.8080
Средний F1-Score: 0.7722
Средний NDCG: 0.8469
Средний Precision: 0.7555
Средний Recall: 0.7930
Средний F1-Score: 0.7589


[I 2026-03-09 15:35:57,271] Trial 28 finished with value: 0.856808336447051 and parameters: {'n_estimators': 650, 'max_depth': 10, 'learning_rate': 0.21007336361673143, 'min_child_weight': 9}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8568
Средний Precision: 0.7633
Средний Recall: 0.7968
Средний F1-Score: 0.7646
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/1/runs/09bc4fd2150b47be88836f8d32c58a97
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8602
Средний Precision: 0.7642
Средний Recall: 0.8052
Средний F1-Score: 0.7687
Средний NDCG: 0.8478
Средний Precision: 0.7515
Средний Recall: 0.7896
Средний F1-Score: 0.7547


[I 2026-03-09 15:36:27,797] Trial 29 finished with value: 0.8564128660345299 and parameters: {'n_estimators': 450, 'max_depth': 14, 'learning_rate': 0.09974470540270561, 'min_child_weight': 6}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8564
Средний Precision: 0.7607
Средний Recall: 0.7960
Средний F1-Score: 0.7626
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/1/runs/02f48496b89548f996560b0b3d73e2e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 30
Best params XGBoost: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.09866580252678012, 'min_child_weight': 10}


In [72]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb, precision_xgb, recall_xgb, f1_xgb = calculate_metrics(df_test_xgb)
metrics_xgb = {
    'NDCG': ndcg_xgb,
    'Precision': precision_xgb,
    'Recall': recall_xgb,
    'F1': f1_xgb
}

Средний NDCG: 0.7792
Средний Precision: 0.6968
Средний Recall: 0.7411
Средний F1-Score: 0.7054


In [73]:
RUN_NAME_XGB = "best_optuna_xgb"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb)
    mlflow.log_params(best_params_xgb)

Registered model 'best_optuna_xgb' already exists. Creating a new version of this model...
2026/03/09 15:36:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb, version 4


🏃 View run best_optuna_xgb at: http://127.0.0.1:5000/#/experiments/1/runs/2deb4b24811947dfa541ac4294bf1096
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_xgb'.


# LGBMClassifier

In [74]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [75]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/808ae0d03de74ee29ab671cdd767a0ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [76]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [77]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=True)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=10, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-03-09 15:36:42,078] Using an existing study with name 'LGBMClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8619
Средний Precision: 0.7401
Средний Recall: 0.8295
Средний F1-Score: 0.7644
Средний NDCG: 0.8495
Средний Precision: 0.7325
Средний Recall: 0.8160
Средний F1-Score: 0.7551


[I 2026-03-09 15:37:22,778] Trial 20 finished with value: 0.8580217559019986 and parameters: {'n_estimators': 550, 'max_depth': 13, 'learning_rate': 0.0762402541615849, 'num_leaves': 60, 'min_child_samples': 58}. Best is trial 20 with value: 0.8580217559019986.


Средний NDCG: 0.8580
Средний Precision: 0.7416
Средний Recall: 0.8218
Средний F1-Score: 0.7631
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/1/runs/bcb51d821cbd4731a5f8a2b35cf13394
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8626
Средний Precision: 0.7426
Средний Recall: 0.8305
Средний F1-Score: 0.7668
Средний NDCG: 0.8487
Средний Precision: 0.7279
Средний Recall: 0.8187
Средний F1-Score: 0.7535


[I 2026-03-09 15:38:03,484] Trial 21 finished with value: 0.8583427584888133 and parameters: {'n_estimators': 600, 'max_depth': 13, 'learning_rate': 0.07129136152747409, 'num_leaves': 57, 'min_child_samples': 57}. Best is trial 21 with value: 0.8583427584888133.


...

Средний NDCG: 0.8570
Средний Precision: 0.7628
Средний Recall: 0.7988
Средний F1-Score: 0.7661
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/1/runs/2facf60285f74f18a98899dfaee73fd7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8608
Средний Precision: 0.7302
Средний Recall: 0.8302
Средний F1-Score: 0.7591
Средний NDCG: 0.8487
Средний Precision: 0.7258
Средний Recall: 0.8189
Средний F1-Score: 0.7522


[I 2026-03-09 15:44:46,881] Trial 28 finished with value: 0.8577983959457124 and parameters: {'n_estimators': 750, 'max_depth': 14, 'learning_rate': 0.019801885923759432, 'num_leaves': 143, 'min_child_samples': 62}. Best is trial 21 with value: 0.8583427584888133.


Средний NDCG: 0.8578
Средний Precision: 0.7323
Средний Recall: 0.8242
Средний F1-Score: 0.7585
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/1/runs/52754b74e51c46a18b1f19c9f15c907e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8620
Средний Precision: 0.7515
Средний Recall: 0.8256
Средний F1-Score: 0.7703
Средний NDCG: 0.8489
Средний Precision: 0.7414
Средний Recall: 0.8123
Средний F1-Score: 0.7595


[I 2026-03-09 15:45:47,450] Trial 29 finished with value: 0.8582549743856155 and parameters: {'n_estimators': 850, 'max_depth': 15, 'learning_rate': 0.04069897253433466, 'num_leaves': 94, 'min_child_samples': 41}. Best is trial 21 with value: 0.8583427584888133.


Средний NDCG: 0.8583
Средний Precision: 0.7481
Средний Recall: 0.8161
Средний F1-Score: 0.7652
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/1/runs/058ca1e201df4ff0b4813391ea74536d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 30
Best params LGBM: {'n_estimators': 600, 'max_depth': 13, 'learning_rate': 0.07129136152747409, 'num_leaves': 57, 'min_child_samples': 57}


In [78]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm, precision_lgbm, recall_lgbm, f1_lgbm = calculate_metrics(df_test_lgbm)
metrics_lgbm = {
    'NDCG': ndcg_lgbm,
    'Precision': precision_lgbm,
    'Recall': recall_lgbm,
    'F1': f1_lgbm
}

Средний NDCG: 0.7799
Средний Precision: 0.6742
Средний Recall: 0.7525
Средний F1-Score: 0.6957


In [79]:
RUN_NAME_LGBM = "best_optuna_lgbm"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm)
    mlflow.log_params(best_params_lgbm)

Registered model 'best_optuna_lgbm' already exists. Creating a new version of this model...
2026/03/09 15:46:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lgbm, version 4


🏃 View run best_optuna_lgbm at: http://127.0.0.1:5000/#/experiments/1/runs/52e1e8895b734184b84038788b2f211b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '4' of model 'best_optuna_lgbm'.


# CatBoostClassifier

In [80]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [81]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

🏃 View run CatBoostClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/fcbc4056414b45e3afd0d34bd6d8f8de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [82]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [83]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=10, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

[I 2026-03-09 15:46:04,747] Using an existing study with name 'CatBoostClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8596
Средний Precision: 0.7001
Средний Recall: 0.8371
Средний F1-Score: 0.7415
Средний NDCG: 0.8480
Средний Precision: 0.6923
Средний Recall: 0.8246
Средний F1-Score: 0.7318


[I 2026-03-09 15:47:01,053] Trial 21 finished with value: 0.8557685661705835 and parameters: {'iterations': 750, 'depth': 8, 'learning_rate': 0.04228185748180876, 'l2_leaf_reg': 3.3791370812095}. Best is trial 19 with value: 0.8575655602470745.


Средний NDCG: 0.8558
Средний Precision: 0.7048
Средний Recall: 0.8315
Средний F1-Score: 0.7427
🏃 View run 21 at: http://127.0.0.1:5000/#/experiments/1/runs/dcf8b5c4545d43558e6412c1c2ed92f0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8614
Средний Precision: 0.7334
Средний Recall: 0.8317
Средний F1-Score: 0.7620
Средний NDCG: 0.8499
Средний Precision: 0.7251
Средний Recall: 0.8177
Средний F1-Score: 0.7510


[I 2026-03-09 15:48:16,817] Trial 22 finished with value: 0.8569256610745281 and parameters: {'iterations': 1000, 'depth': 9, 'learning_rate': 0.06541320668588411, 'l2_leaf_reg': 5.2317028648453165}. Best is trial 19 with value: 0.8575655602470745.


...

Средний NDCG: 0.8570
Средний Precision: 0.7128
Средний Recall: 0.8279
Средний F1-Score: 0.7467
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/1/runs/3e90d4dad3964868a08912b54ba85aad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8597
Средний Precision: 0.7031
Средний Recall: 0.8379
Средний F1-Score: 0.7440
Средний NDCG: 0.8482
Средний Precision: 0.6928
Средний Recall: 0.8251
Средний F1-Score: 0.7325


[I 2026-03-09 15:56:59,432] Trial 29 finished with value: 0.8565102130012653 and parameters: {'iterations': 900, 'depth': 8, 'learning_rate': 0.039107547820079414, 'l2_leaf_reg': 7.152410258853395}. Best is trial 19 with value: 0.8575655602470745.


Средний NDCG: 0.8565
Средний Precision: 0.7046
Средний Recall: 0.8316
Средний F1-Score: 0.7427
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/1/runs/f4766e8e78c443aeb2af90bf08e87221
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8613
Средний Precision: 0.7455
Средний Recall: 0.8274
Средний F1-Score: 0.7675
Средний NDCG: 0.8491
Средний Precision: 0.7412
Средний Recall: 0.8079
Средний F1-Score: 0.7568


[I 2026-03-09 15:58:21,985] Trial 30 finished with value: 0.8571390730299547 and parameters: {'iterations': 700, 'depth': 10, 'learning_rate': 0.11142421102899439, 'l2_leaf_reg': 3.8781777861829982}. Best is trial 19 with value: 0.8575655602470745.


Средний NDCG: 0.8571
Средний Precision: 0.7475
Средний Recall: 0.8154
Средний F1-Score: 0.7643
🏃 View run 30 at: http://127.0.0.1:5000/#/experiments/1/runs/4b922e1e13894283a080a70e81e4e4e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 31
Best params CatBoost: {'iterations': 750, 'depth': 9, 'learning_rate': 0.049337484129647155, 'l2_leaf_reg': 5.635559637911369}


In [84]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost, precision_catboost, recall_catboost, f1_catboost = calculate_metrics(df_test_catboost)
metrics_catboost = {
    'NDCG': ndcg_catboost,
    'Precision': precision_catboost,
    'Recall': recall_catboost,
    'F1': f1_catboost
}

Средний NDCG: 0.7779
Средний Precision: 0.6595
Средний Recall: 0.7554
Средний F1-Score: 0.6875


In [85]:
RUN_NAME_CATBOOST = "best_optuna_catboost"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost)
    mlflow.log_params(best_params_catboost)

Registered model 'best_optuna_catboost' already exists. Creating a new version of this model...
2026/03/09 15:58:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost, version 5


🏃 View run best_optuna_catboost at: http://127.0.0.1:5000/#/experiments/1/runs/50736e5e547d43a69fb3cdcf31982061
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '5' of model 'best_optuna_catboost'.


# Model comparison

In [86]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost'],
    'NDCG': [ndcg_lr, ndcg_rf, 
             ndcg_xgb, ndcg_lgbm, ndcg_catboost],
    'Precision': [precision_lr, precision_rf, 
                  precision_xgb, precision_lgbm, precision_catboost],
    'Recall': [recall_lr, recall_rf, 
               recall_xgb, recall_lgbm, recall_catboost],
    'F1': [f1_lr, f1_rf, 
           f1_xgb, f1_lgbm, f1_catboost]
})

models_comparison

,Model,NDCG,Precision,Recall,F1
0,LogisticRegression,0.751483,0.638633,0.598481,0.602124
1,RandomForest,0.775498,0.656790,0.746367,0.681294
2,XGBoost,0.779192,0.696847,0.741136,0.705449
3,LightGBM,0.779915,0.674173,0.752533,0.695656
4,CatBoost,0.777929,0.659482,0.755417,0.687453


In [87]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")


Лучшая модель: LightGBM с NDCG = 0.7799


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

По итогам подбора гиперпараметров лучшее качество показала модель `XGBClassifier`. Теперь попробуем добавить дополнительные признаки от кандидата генераторов. В первую очередь используем коллаборативную фильтрацию.

</div>

# ALS

## Feature engineering

In [88]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes = df['resume_id'].unique().tolist()

    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id = {v: k for k, v in id2resume.items()}

    rows = [vacancy2id[vacancy] for vacancy in df['vacancy_id']]
    cols = [resume2id[resume] for resume in df['resume_id']]

    interactions_matrix = csr_matrix(
        (df['target'], (rows, cols)),
        shape=(len(unique_vacancies), len(unique_resumes)),
        dtype=np.float32,
    )

    return interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

In [89]:
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(df)

print(f"Размерность матрицы взаимодействий: {interactions_matrix.shape}")
print(f"Плотность матрицы: {interactions_matrix.nnz / (interactions_matrix.shape[0] * interactions_matrix.shape[1]):.6f}")

Размерность матрицы взаимодействий: (3409, 20130)
Плотность матрицы: 0.004687


In [90]:
def get_factors(interactions_matrix):
    als_model = AlternatingLeastSquares(
        factors=64,          
        regularization=0.1,  
        iterations=30,       
        random_state=RANDOM_STATE,
        num_threads=0
    )
    
    als_model.fit(interactions_matrix.T)

    vacancy_factors = als_model.item_factors
    resume_factors = als_model.user_factors

    return vacancy_factors, resume_factors

In [91]:
vacancy_factors, resume_factors = get_factors(interactions_matrix)

print(f"Размерность эмбеддингов вакансий: {vacancy_factors.shape}")
print(f"Размерность эмбеддингов резюме: {resume_factors.shape}")

  0%|          | 0/30 [00:00<?, ?it/s]

Размерность эмбеддингов вакансий: (3409, 64)
Размерность эмбеддингов резюме: (20130, 64)


In [92]:
def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    else:
        vac_idx = vacancy2id[vacancy_id]
        res_idx = resume2id[resume_id]
        score = np.dot(vacancy_factors[vac_idx], resume_factors[res_idx])
        return score

df['als_score'] = df.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

In [93]:
df[['vacancy_id', 'resume_id', 'target', 'als_score']].head()

,vacancy_id,resume_id,target,als_score
0,126167948,6969174,1,0.000042
1,126167948,9100077,1,0.000033
2,126167948,32644957,1,0.000019
3,126167948,27220466,1,0.000022
4,126167948,7532708,1,0.000021


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Проверим схожесть резюме по латентным векторам.

</div>

In [94]:
def find_similar_resumes(resume_id, resume_factors, n_similar=10, metric='cosine'):
    """
    Находит N наиболее похожих резюме на заданное.
    """
    if resume_id not in resume2id:
        print(f"Резюме с ID {resume_id} не найдено")
        return []

    target_idx = resume2id[resume_id]
    target_vector = resume_factors[target_idx].reshape(1, -1)
    
    if metric == 'cosine':
        similarities = cosine_similarity(target_vector, resume_factors)[0]
    else:
        similarities = np.dot(resume_factors, target_vector.T).flatten()
    
    similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]
    
    similar_resumes = []
    for idx in similar_indices:
        sim_resume_id = unique_resumes[idx]
        similarity_score = similarities[idx]
        similar_resumes.append((sim_resume_id, similarity_score))
    
    return similar_resumes

def get_resume_profile(resume_id, df):
    """Вспомогательная функция для получения информации о резюме"""
    resume_data = df[df['resume_id'] == resume_id].iloc[0]
    return {
        'resume_id': resume_id,
        'title': resume_data.get('resume_title', 'N/A'),
        'specialization': resume_data.get('resume_specialization', 'N/A'),
        'skills': resume_data.get('resume_skills', 'N/A'),
        'experience': resume_data.get('resume_total_experience', 'N/A'),
        'salary': resume_data.get('resume_salary', 'N/A'),
        'location': resume_data.get('resume_location', 'N/A')
    }

In [95]:
example_resume_id = df['resume_id'].sample(1).values[0].item()
print(f"Исходное резюме (ID: {example_resume_id}):")
original_profile = get_resume_profile(example_resume_id, df)
for key, value in original_profile.items():
    print(f"  {key}: {value}")

print(f"\nТоп-5 похожих резюме (косинусная близость):")
similar_resumes = find_similar_resumes(example_resume_id, resume_factors, n_similar=5, metric='cosine')

for i, (sim_id, score) in enumerate(similar_resumes, 1):
    profile = get_resume_profile(sim_id, df)
    print(f"\n{i}. Резюме ID: {sim_id} (сходство: {score:.4f})")
    print(f"   Должность: {profile['title']}")
    print(f"   Специализация: {profile['specialization']}")
    print(f"   Локация: {profile['location']}")
    print(f"   Опыт: {profile['experience']}")

Исходное резюме (ID: 65353364):
  resume_id: 65353364
  title: Системный администратор
  specialization: ['Сетевой инженер', 'Системный администратор', 'Системный инженер']
  skills: [' Linux', 'Основы анализа данных.', 'Основы Python', 'Информационные технологии', 'Администрирование сетевого оборудования', 'Сетевые технологии', 'Active Directory', 'VLAN', 'OSPF', 'Системное администрирование', 'Техническая поддержка', 'Cisco', 'Qtech', 'Huawei', 'Eltex', 'Основы SQL', 'Helpdesk', 'Cloud', 'OSI', 'Администрирование серверов Linux', 'Zabbix']
  experience: 15 лет 10 месяцев
  salary: 170000.0
  location: Москва

Топ-5 похожих резюме (косинусная близость):

1. Резюме ID: 57082774 (сходство: 0.0000)
   Должность: Директор по маркетингу
   Специализация: ['Директор по маркетингу и PR (CMO)', 'Руководитель отдела маркетинга и рекламы']
   Локация: Санкт-Петербург
   Опыт: 8 лет 11 месяцев

2. Резюме ID: 148646788 (сходство: 0.0000)
   Должность: Веб-разработчик
   Специализация: ['Программи

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Разделим тренировочную выборку еще на две, чтобы избежать подглядывания и добавить в признак `als_score` нулевые значения в тренировочный датасет, т.к. у нас могут быть вакансии и резюме без взаимодействий до обучения.

</div>

In [96]:
X_train_als, X_test_als, y_train_als, y_test_als = train_test_split(X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

In [97]:
train_als_interactions = df.loc[X_train_als.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_train[['vacancy_id', 'resume_id']] = df.loc[X_train.index, ['vacancy_id', 'resume_id']]
X_train['als_score'] = X_train.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_train = X_train.drop(['vacancy_id', 'resume_id'], axis=1)

  0%|          | 0/30 [00:00<?, ?it/s]

In [98]:
# проверим есть ли вакансии и резюме без взаимодействий до обучения
X_train[X_train['als_score'] == 0]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf,als_score
286115,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,40000.0,38.0,223.0,Москва,Неизвестно,NDT,208.0,1,0,0.25,0.015081,0.0
216165,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.038536,0.0
297591,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.016496,0.0
192099,Москва,Более 6 лет,Полная занятость,Полный день,30000.0,57.0,289.0,Москва,Мужчина,NDT,170.0,1,0,0.00,0.000000,0.0
82523,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,0.0,38.0,185.0,Москва,Мужчина,NDT,114.0,1,4,0.00,0.063418,0.0
32590,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,38.0,185.0,Москва,Мужчина,NDT,114.0,1,2,0.00,0.101719,0.0
260861,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,30000.0,57.0,289.0,Москва,Мужчина,NDT,170.0,1,0,0.00,0.020658,0.0
52865,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.011811,0.0
175258,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.021698,0.0
217714,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.012118,0.0


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь добавим признак `als_score` для тестовой выборке, но уже возьмем матрицу интеракций на полной тренировочной выборке.

</div>

In [99]:
train_interactions = df.loc[X_train.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_test[['vacancy_id', 'resume_id']] = df.loc[X_test.index, ['vacancy_id', 'resume_id']]
X_test['als_score'] = X_test.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_test = X_test.drop(['vacancy_id', 'resume_id'], axis=1)

  0%|          | 0/30 [00:00<?, ?it/s]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Выборки получены, обучим `XGBClassifier` с подбором гиперпараметров.

</div>

## XGBClassifier

In [121]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [122]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/5ea785191f4547fdb3ad318463cf8826
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [123]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME_XGB = "XGBClassifier_optuna_als"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [124]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=True)

study_xgb.optimize(objective_xgb, 
                   n_trials=10, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-03-09 16:31:45,122] Using an existing study with name 'XGBClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8680
Средний Precision: 0.7971
Средний Recall: 0.8364
Средний F1-Score: 0.8067
Средний NDCG: 0.8583
Средний Precision: 0.7880
Средний Recall: 0.8272
Средний F1-Score: 0.7970


[I 2026-03-09 16:32:19,434] Trial 20 finished with value: 0.8632976822281608 and parameters: {'n_estimators': 950, 'max_depth': 8, 'learning_rate': 0.04198103143454717, 'min_child_weight': 4}. Best is trial 17 with value: 0.863351964970977.


Средний NDCG: 0.8633
Средний Precision: 0.7920
Средний Recall: 0.8302
Средний F1-Score: 0.8003
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/1/runs/83b443c5b086498aa945281039b14baf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8677
Средний Precision: 0.7968
Средний Recall: 0.8374
Средний F1-Score: 0.8067
Средний NDCG: 0.8577
Средний Precision: 0.7862
Средний Recall: 0.8278
Средний F1-Score: 0.7960


[I 2026-03-09 16:32:51,800] Trial 21 finished with value: 0.8632052895317053 and parameters: {'n_estimators': 950, 'max_depth': 8, 'learning_rate': 0.04067316308182902, 'min_child_weight': 4}. Best is trial 17 with value: 0.863351964970977.


...

Средний NDCG: 0.8634
Средний Precision: 0.7955
Средний Recall: 0.8270
Средний F1-Score: 0.8007
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/1/runs/67b2fcfc4db44fc1a949afc50ec820dc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8670
Средний Precision: 0.7964
Средний Recall: 0.8321
Средний F1-Score: 0.8041
Средний NDCG: 0.8578
Средний Precision: 0.7910
Средний Recall: 0.8266
Средний F1-Score: 0.7983


[I 2026-03-09 16:36:33,212] Trial 28 finished with value: 0.863151002642667 and parameters: {'n_estimators': 700, 'max_depth': 11, 'learning_rate': 0.02892070398572207, 'min_child_weight': 5}. Best is trial 26 with value: 0.8635372074877529.


Средний NDCG: 0.8632
Средний Precision: 0.7930
Средний Recall: 0.8276
Средний F1-Score: 0.7996
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/1/runs/4acf31b20f6b40a39656a4da2a274221
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8679
Средний Precision: 0.7814
Средний Recall: 0.8406
Средний F1-Score: 0.7990
Средний NDCG: 0.8583
Средний Precision: 0.7741
Средний Recall: 0.8330
Средний F1-Score: 0.7909


[I 2026-03-09 16:37:03,489] Trial 29 finished with value: 0.8622978119795549 and parameters: {'n_estimators': 550, 'max_depth': 10, 'learning_rate': 0.016034055878559182, 'min_child_weight': 6}. Best is trial 26 with value: 0.8635372074877529.


Средний NDCG: 0.8623
Средний Precision: 0.7793
Средний Recall: 0.8338
Средний F1-Score: 0.7939
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/1/runs/84150ab13f2244889766dfb74af438ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 30
Best params XGBoost: {'n_estimators': 750, 'max_depth': 8, 'learning_rate': 0.03575111265940126, 'min_child_weight': 3}


In [125]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb_als, precision_xgb_als, recall_xgb_als, f1_xgb_als = calculate_metrics(df_test_xgb)
metrics_xgb_als = {
    'NDCG': ndcg_xgb_als,
    'Precision': precision_xgb_als,
    'Recall': recall_xgb_als,
    'F1': f1_xgb_als
}

Средний NDCG: 0.7807
Средний Precision: 0.7041
Средний Recall: 0.7496
Средний F1-Score: 0.7145


In [126]:
RUN_NAME_XGB = "best_optuna_xgb_als"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb_als',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb_als)
    mlflow.log_params(best_params_xgb)

Registered model 'best_optuna_xgb_als' already exists. Creating a new version of this model...
2026/03/09 16:37:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb_als, version 3


🏃 View run best_optuna_xgb_als at: http://127.0.0.1:5000/#/experiments/1/runs/bc663df7de8d4aeaae1c9c6edabc66e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '3' of model 'best_optuna_xgb_als'.


## LGBMClassifier

In [127]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [128]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/dd01cf77148e4b4581e8b03581a5dd77
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [129]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna_als"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [130]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=True)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=10, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-03-09 16:37:18,908] Using an existing study with name 'LGBMClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8687
Средний Precision: 0.7729
Средний Recall: 0.8489
Средний F1-Score: 0.7970
Средний NDCG: 0.8579
Средний Precision: 0.7617
Средний Recall: 0.8390
Средний F1-Score: 0.7862


[I 2026-03-09 16:37:58,314] Trial 20 finished with value: 0.8639873317033421 and parameters: {'n_estimators': 950, 'max_depth': 13, 'learning_rate': 0.040298322649030646, 'num_leaves': 22, 'min_child_samples': 59}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8640
Средний Precision: 0.7736
Средний Recall: 0.8425
Средний F1-Score: 0.7944
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/1/runs/3b9ec3816d114f748239cfcd1a4d8e2b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8682
Средний Precision: 0.7907
Средний Recall: 0.8445
Средний F1-Score: 0.8059
Средний NDCG: 0.8579
Средний Precision: 0.7793
Средний Recall: 0.8333
Средний F1-Score: 0.7944


[I 2026-03-09 16:38:49,971] Trial 21 finished with value: 0.8631772431472983 and parameters: {'n_estimators': 950, 'max_depth': 13, 'learning_rate': 0.038094492901624956, 'num_leaves': 44, 'min_child_samples': 57}. Best is trial 19 with value: 0.8642042129240123.


...

Средний NDCG: 0.8639
Средний Precision: 0.7659
Средний Recall: 0.8435
Средний F1-Score: 0.7902
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/1/runs/f7104da6332c4fce99e659506f65b6a6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8677
Средний Precision: 0.8067
Средний Recall: 0.8230
Средний F1-Score: 0.8057
Средний NDCG: 0.8576
Средний Precision: 0.7998
Средний Recall: 0.8170
Средний F1-Score: 0.7989


[I 2026-03-09 16:44:34,438] Trial 28 finished with value: 0.863640926039298 and parameters: {'n_estimators': 850, 'max_depth': 14, 'learning_rate': 0.07286300213248183, 'num_leaves': 102, 'min_child_samples': 41}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8636
Средний Precision: 0.8042
Средний Recall: 0.8178
Средний F1-Score: 0.8009
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/1/runs/b6016f77039947a4a531dcf9d25992b4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8681
Средний Precision: 0.7991
Средний Recall: 0.8365
Средний F1-Score: 0.8074
Средний NDCG: 0.8580
Средний Precision: 0.7904
Средний Recall: 0.8247
Средний F1-Score: 0.7965


[I 2026-03-09 16:45:27,683] Trial 29 finished with value: 0.8632768035147501 and parameters: {'n_estimators': 1000, 'max_depth': 15, 'learning_rate': 0.04953793850984789, 'num_leaves': 55, 'min_child_samples': 79}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8633
Средний Precision: 0.7942
Средний Recall: 0.8284
Средний F1-Score: 0.8006
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/1/runs/21f2a74fd5994d4095067afec047d0b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 30
Best params LGBM: {'n_estimators': 900, 'max_depth': 13, 'learning_rate': 0.04019165012583699, 'num_leaves': 23, 'min_child_samples': 54}


In [131]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm_als, precision_lgbm_als, recall_lgbm_als, f1_lgbm_als = calculate_metrics(df_test_lgbm)
metrics_lgbm_als = {
    'NDCG': ndcg_lgbm_als,
    'Precision': precision_lgbm_als,
    'Recall': recall_lgbm_als,
    'F1': f1_lgbm_als
}

Средний NDCG: 0.7816
Средний Precision: 0.6869
Средний Recall: 0.7583
Средний F1-Score: 0.7072


In [132]:
RUN_NAME_LGBM = "best_optuna_lgbm_als"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm_als',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm_als)
    mlflow.log_params(best_params_lgbm)

Registered model 'best_optuna_lgbm_als' already exists. Creating a new version of this model...
2026/03/09 16:45:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lgbm_als, version 3


🏃 View run best_optuna_lgbm_als at: http://127.0.0.1:5000/#/experiments/1/runs/6d328e8dccdf444c9ee4f754844635fe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '3' of model 'best_optuna_lgbm_als'.


## CatBoostClassifier

In [133]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [134]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

🏃 View run CatBoostClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/fbbdf4f8bbb648da8dd0a984e9a15b73
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [135]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna_als"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [136]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=10, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

[I 2026-03-09 16:45:44,503] Using an existing study with name 'CatBoostClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8678
Средний Precision: 0.7434
Средний Recall: 0.8542
Средний F1-Score: 0.7798
Средний NDCG: 0.8582
Средний Precision: 0.7336
Средний Recall: 0.8440
Средний F1-Score: 0.7702


[I 2026-03-09 16:46:42,537] Trial 20 finished with value: 0.8633344410676325 and parameters: {'iterations': 750, 'depth': 7, 'learning_rate': 0.017256931110882744, 'l2_leaf_reg': 3.3472028130345293}. Best is trial 12 with value: 0.863972997356322.


Средний NDCG: 0.8633
Средний Precision: 0.7441
Средний Recall: 0.8483
Средний F1-Score: 0.7788
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/1/runs/00e51613206f45b89a31e9dcac8cc865
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8678
Средний Precision: 0.7840
Средний Recall: 0.8456
Средний F1-Score: 0.8027
Средний NDCG: 0.8579
Средний Precision: 0.7718
Средний Recall: 0.8367
Средний F1-Score: 0.7918


[I 2026-03-09 16:48:24,097] Trial 21 finished with value: 0.8633394014425736 and parameters: {'iterations': 900, 'depth': 10, 'learning_rate': 0.05853170067668366, 'l2_leaf_reg': 5.790843642561652}. Best is trial 12 with value: 0.863972997356322.


...

Средний NDCG: 0.8635
Средний Precision: 0.7828
Средний Recall: 0.8389
Средний F1-Score: 0.7988
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/1/runs/94fbc996527b4e9eb0738da9b3ad88c0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8676
Средний Precision: 0.7769
Средний Recall: 0.8477
Средний F1-Score: 0.7987
Средний NDCG: 0.8570
Средний Precision: 0.7702
Средний Recall: 0.8369
Средний F1-Score: 0.7906


[I 2026-03-09 16:57:07,770] Trial 28 finished with value: 0.8632602517371512 and parameters: {'iterations': 700, 'depth': 7, 'learning_rate': 0.17646596150449267, 'l2_leaf_reg': 9.972130137386895}. Best is trial 26 with value: 0.8640644662917454.


Средний NDCG: 0.8633
Средний Precision: 0.7779
Средний Recall: 0.8424
Средний F1-Score: 0.7976
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/1/runs/67ac8319f2d4417d9986c40a583cba44
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8677
Средний Precision: 0.7922
Средний Recall: 0.8398
Средний F1-Score: 0.8052
Средний NDCG: 0.8577
Средний Precision: 0.7798
Средний Recall: 0.8331
Средний F1-Score: 0.7949
Средний NDCG: 0.8639
Средний Precision: 0.7843
Средний Recall: 0.8355
Средний F1-Score: 0.7980


[I 2026-03-09 16:58:11,012] Trial 29 finished with value: 0.8638996139433601 and parameters: {'iterations': 450, 'depth': 10, 'learning_rate': 0.10915971005005015, 'l2_leaf_reg': 3.970392945279382}. Best is trial 26 with value: 0.8640644662917454.


🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/1/runs/afc27276e3a14378a878f9ed1049163e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 30
Best params CatBoost: {'iterations': 900, 'depth': 10, 'learning_rate': 0.06997584640069045, 'l2_leaf_reg': 8.193197313867714}


In [137]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost_als, precision_catboost_als, recall_catboost_als, f1_catboost_als = calculate_metrics(df_test_catboost)
metrics_catboost_als = {
    'NDCG': ndcg_catboost_als,
    'Precision': precision_catboost_als,
    'Recall': recall_catboost_als,
    'F1': f1_catboost_als
}

Средний NDCG: 0.7816
Средний Precision: 0.7050
Средний Recall: 0.7508
Средний F1-Score: 0.7148


In [138]:
RUN_NAME_CATBOOST = "best_optuna_catboost_als"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost_als',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost_als)
    mlflow.log_params(best_params_catboost)

Registered model 'best_optuna_catboost_als' already exists. Creating a new version of this model...
2026/03/09 16:58:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost_als, version 3


🏃 View run best_optuna_catboost_als at: http://127.0.0.1:5000/#/experiments/1/runs/d9d3ff3459574265918ce47c909f2249
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '3' of model 'best_optuna_catboost_als'.


## Model comparison

In [139]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'XGBoost + ALS', 'LightGBM + ALS', 'CatBoost + ALS'],
    'NDCG': [ndcg_lr, ndcg_rf, ndcg_xgb, ndcg_lgbm, ndcg_catboost, ndcg_xgb_als, ndcg_lgbm_als, ndcg_catboost_als],
    'Precision': [precision_lr, precision_rf, precision_xgb, precision_lgbm, precision_catboost, precision_xgb_als, precision_lgbm_als, precision_catboost_als],
    'Recall': [recall_lr, recall_rf, recall_xgb, recall_lgbm, recall_catboost, recall_xgb_als, recall_lgbm_als, recall_catboost_als],
    'F1': [f1_lr, f1_rf, f1_xgb, f1_lgbm, f1_catboost, f1_xgb_als, f1_lgbm_als, f1_catboost_als]
})

models_comparison

,Model,NDCG,Precision,Recall,F1
0,LogisticRegression,0.751483,0.638633,0.598481,0.602124
1,RandomForest,0.775498,0.656790,0.746367,0.681294
2,XGBoost,0.779192,0.696847,0.741136,0.705449
3,LightGBM,0.779915,0.674173,0.752533,0.695656
4,CatBoost,0.777929,0.659482,0.755417,0.687453
5,XGBoost + ALS,0.780692,0.704120,0.749619,0.714494
6,LightGBM + ALS,0.781578,0.686926,0.758311,0.707198
7,CatBoost + ALS,0.781646,0.705020,0.750807,0.714827


In [140]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")


Лучшая модель: CatBoost + ALS с NDCG = 0.7816


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сохраним обученный итоговый пайплайн.

</div>

In [141]:
MODEL_NAME = 'pipeline_catboost_best.pkl'
with open(MODEL_NAME, 'wb') as file:
    pickle.dump(pipeline_xgb_best, file)